# BioMechAI — PoseC3D v7 (cleaned data, V5 recipe)

Goal: test whether **more, cleaner data** improves on the V5 champion (53.38% top-1), using the *same model and training recipe* as V5.

## What this notebook does
1. **Cleans the data** before training: removes exact duplicate clips, removes train clips that also appear in the val set (same content, same video id, or shared frames), and drops frozen / low-motion / out-of-frame clips. The val set is never filtered.
2. **Trains** the V5 config (SlowOnly-R50 limb heatmaps, FineGYM-pretrained, 48 frames, 56×56) for 18 epochs with a fixed seed and optional class weights.
3. **Evaluates** the *best* and the *last* checkpoint on (a) the 192 unique val clips and (b) the full 444-row frozen val file, with confidence intervals, per-class recall and a confusion matrix.
4. **Packages** checkpoints, config, data report and results into one zip.

## Before you run
- Attach a Kaggle dataset containing **`custom_dataset_train_v7_1.pkl`** (output of `--merge-v7`) and **`custom_dataset_val_frozen.pkl`**. Names can be changed in Cell 3.
- Use a GPU session (T4 x2 or P100).
- Cell 1 mirrors the setup that trained successfully on Kaggle (PyTorch 2.11 / CUDA 12.8, mmcv-lite 2.2.0 fallback, patches applied to both `train.py` and `test.py`).
- **No credentials are needed.** Do not add tokens, secrets or Drive keys to this notebook. Checkpoints are downloaded from the Output tab.

## Run order
Cell 1 → 2 → 3 → 4 → 5 → 6 → 7. Cell 4 stops with a clear error if any class ends up too small or any train/val overlap remains.


## Cell 1: Install libraries & apply patches (run first)

In [1]:
# ======================================================================
# CELL 1: BioMechAI v7 - Fresh Session Setup & Patches
# Mirrors the setup that trained successfully on Kaggle (PyTorch 2.11 / CUDA 12.8):
# no prebuilt mmcv wheel exists for that torch, so it falls back to mmcv-lite 2.2.0.
# No credentials are used anywhere in this notebook.
# ======================================================================
import os, sys, subprocess, functools
from importlib import metadata

print("⏳ Step 1/4: Installing mmengine...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "mmengine==0.10.7"], check=True)

print("⏳ Step 2/4: Installing mmcv (with smart prebuilt wheel fallback)...")
import torch
print(f"   Detected Environment: PyTorch {torch.__version__}, CUDA {torch.version.cuda}")

installed_mmcv = False
# Attempt A / B: OpenMMLab prebuilt wheels (binary only, so pip never tries a source build)
for tag in ("torch2.4", "torch2.1"):
    res = subprocess.run([
        sys.executable, "-m", "pip", "install", "-q", "--only-binary=:all:", "mmcv==2.2.0",
        "-f", f"https://download.openmmlab.com/mmcv/dist/cu121/{tag}/index.html"
    ], capture_output=True, text=True)
    if res.returncode == 0:
        installed_mmcv = True
        print(f"   ✅ Installed prebuilt mmcv 2.2.0 (cu121/{tag})")
        break
    print(f"   ℹ️ No prebuilt mmcv wheel for {tag}; trying next option...")

# Attempt C: mmcv-lite (pure PyPI wheel, no C++ build, enough for PoseC3D transforms)
if not installed_mmcv:
    print("   ℹ️ Falling back to universal mmcv-lite 2.2.0 (avoids C++ compilation mismatch)...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "mmcv-lite==2.2.0"], check=True)
    print("   ✅ Installed mmcv-lite 2.2.0 successfully!")

print("⏳ Step 3/4: Installing supporting packages (einops, timm)...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "einops", "timm"], check=True)

print("⏳ Step 4/4: Cloning MMAction2 Framework from GitHub...")
subprocess.run(["rm", "-rf", "/kaggle/working/mmaction2"], check=False)
subprocess.run([
    "git", "clone", "--depth=1",
    "https://github.com/open-mmlab/mmaction2.git",
    "/kaggle/working/mmaction2"
], check=True)

# Apply patches to avoid setup.py compilation crashes
print("🛠 Applying compatibility patches...")

# Patch 1: MMCV version ceiling guard (allows mmcv 2.2.0 to work)
init_path = "/kaggle/working/mmaction2/mmaction/__init__.py"
if os.path.exists(init_path):
    with open(init_path) as f:
        content = f.read()
    new_content = content.replace("mmcv_maximum_version = '2.2.0'", "mmcv_maximum_version = '2.3.0'")
    with open(init_path, "w") as f:
        f.write(new_content)
    print("✅ Patch 1: MMCV ceiling guard set to 2.3.0")

# Patch 2: Disable unused multimodal module (avoids PIL/HuggingFace conflicts)
mm_init = "/kaggle/working/mmaction2/mmaction/models/multimodal/__init__.py"
if os.path.exists(os.path.dirname(mm_init)):
    os.makedirs(os.path.dirname(mm_init), exist_ok=True)
    with open(mm_init, "w") as f:
        f.write("__all__ = []\n")
    print("✅ Patch 2: Multimodal module disabled")

# Patch 3: np.Inf -> np.inf (NumPy 2.0 compatibility)
patched_files = 0
for root, dirs, files in os.walk("/kaggle/working/mmaction2"):
    for fname in files:
        if fname.endswith(".py"):
            fpath = os.path.join(root, fname)
            try:
                with open(fpath) as f:
                    txt = f.read()
                if "np.Inf" in txt:
                    with open(fpath, "w") as f:
                        f.write(txt.replace("np.Inf", "np.inf"))
                    patched_files += 1
            except Exception:
                pass
print(f"✅ Patch 3: np.Inf patched in {patched_files} files")

# Patch 4: PyTorch 2.6+ weights_only & C++ ABI mismatch in tools/train.py AND tools/test.py
# (test.py is patched too, because Cell 6 evaluates checkpoints with it)
patch_header = """import functools
import torch as _torch
_orig_torch_load = _torch.load
_torch.load = functools.partial(_orig_torch_load, weights_only=False)
try:
    import mmengine.model.utils as _mu
    _mu.mmcv_full_available = lambda: False
except Exception:
    pass
"""
for tool in ("train.py", "test.py"):
    tool_path = f"/kaggle/working/mmaction2/tools/{tool}"
    if os.path.exists(tool_path):
        with open(tool_path) as f:
            tool_content = f.read()
        if "weights_only=False" not in tool_content:
            with open(tool_path, "w") as f:
                f.write(patch_header + tool_content)
            print(f"✅ Patch 4: tools/{tool} patched")

# Add mmaction2 directly to sys.path so no pip install -e is required
if "/kaggle/working/mmaction2" not in sys.path:
    sys.path.insert(0, "/kaggle/working/mmaction2")
os.environ["PYTHONPATH"] = "/kaggle/working/mmaction2:" + os.environ.get("PYTHONPATH", "")

def _ver(*names):
    for n in names:
        try:
            return f"{n} {metadata.version(n)}"
        except Exception:
            pass
    return "not found"

print("\n" + "=" * 65)
print("🎉 Cell 1 Complete: Environment ready for training!")
print(f"   PyTorch {torch.__version__} | CUDA available: {torch.cuda.is_available()}")
print(f"   {_ver('mmengine')} | {_ver('mmcv', 'mmcv-lite')}")
print("=" * 65)

⏳ Step 1/4: Installing mmengine...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.8/46.8 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 452.7/452.7 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 256.2/256.2 kB 13.8 MB/s eta 0:00:00
⏳ Step 2/4: Installing mmcv (with smart prebuilt wheel fallback)...
   Detected Environment: PyTorch 2.11.0+cu128, CUDA 12.8
   ℹ️ No prebuilt mmcv wheel for torch2.4; trying next option...
   ℹ️ No prebuilt mmcv wheel for torch2.1; trying next option...
   ℹ️ Falling back to universal mmcv-lite 2.2.0 (avoids C++ compilation mismatch)...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 732.3/732.3 kB 23.5 MB/s eta 0:00:00
   ✅ Installed mmcv-lite 2.2.0 successfully!
⏳ Step 3/4: Installing supporting packages (einops, timm)...
⏳ Step 4/4: Cloning MMAction2 Framework from GitHub...


Cloning into '/kaggle/working/mmaction2'...


🛠 Applying compatibility patches...
✅ Patch 1: MMCV ceiling guard set to 2.3.0
✅ Patch 2: Multimodal module disabled
✅ Patch 3: np.Inf patched in 7 files
✅ Patch 4: tools/train.py patched
✅ Patch 4: tools/test.py patched

🎉 Cell 1 Complete: Environment ready for training!
   PyTorch 2.11.0+cu128 | CUDA available: True
   mmengine 0.10.7 | mmcv-lite 2.2.0


## Cell 2: Hardware check

In [2]:
# ======================================================================
# CELL 2: Hardware check
# ======================================================================
import torch

print("=" * 65)
print("HARDWARE CHECK")
print("=" * 65)
if torch.cuda.is_available():
    print(f"✅ GPU Detected: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
else:
    print("⚠️ No GPU detected! Please change Kaggle Accelerator to GPU T4 x2 or P100.")

HARDWARE CHECK
✅ GPU Detected: Tesla T4
   VRAM: 14.56 GB


## Cell 3: Experiment settings

In [3]:
# ======================================================================
# CELL 3: EXPERIMENT SETTINGS  (edit only this cell)
# ======================================================================
import os

EXPERIMENT_NAME = "v7_dedup"        # change if you vary settings below (e.g. "v7_frac50")

# --- where to look for the data (Kaggle: attach your dataset to the notebook) ---
SEARCH_ROOTS    = ["/kaggle/input", "/kaggle/working"]
WORK_ROOT       = "/kaggle/working"
TRAIN_FILE_NAME = "custom_dataset_train_v7.pkl"    # output of --merge-v7 (final Batch 3 run)
VAL_FILE_NAME   = "custom_dataset_val_frozen.pkl"    # the frozen benchmark clips

# --- data cleaning (train set only; the val set is never filtered) ---
NEAR_DUP_MIN_SHARED_FRAMES = 20     # drop train clips sharing >= N identical frames with a val clip
APPLY_QUALITY_FILTER = True
MIN_MOTION       = 5.0              # temporal std of keypoints (0-1000 coordinate scale)
MIN_MOTION_OVERRIDES = {"plank": 1.0}   # plank is a hold, so it legitimately moves very little
MAX_FAKE_FRAMES  = 30               # max copied / placeholder / very-low-visibility frames (of 90)
MAX_OOB_FRACTION = 0.25             # max fraction of keypoints far outside the frame (0.05 wiped out half of bicep_curl)
MAX_CLIPS_PER_VIDEO = None          # e.g. 6 to cap clips per video; None = no cap
TRAIN_FRACTION   = 1.0              # <1.0 keeps that share of videos per class (learning-curve test)
USE_CLASS_WEIGHTS = True            # inverse-frequency class weights in the loss
MIN_CLIPS_PER_CLASS = 30            # safety stop if a class ends up too small

# --- training (same recipe as V5 unless noted) ---
MAX_EPOCHS   = 18
VAL_INTERVAL = 2
BATCH_SIZE   = 16
LR           = 0.01
SEED         = 42                   # new in v7: makes runs repeatable

# --- derived paths ---
DATA_DIR   = os.path.join(WORK_ROOT, f"data_clean_{EXPERIMENT_NAME}")
WORK_DIR   = os.path.join(WORK_ROOT, f"work_dirs/posec3d_{EXPERIMENT_NAME}")
CONFIG_PY  = os.path.join(WORK_ROOT, f"posec3d_biomechai_{EXPERIMENT_NAME}.py")
ZIP_PATH   = os.path.join(WORK_ROOT, f"BioMechAI_PoseC3D_{EXPERIMENT_NAME}_artifacts.zip")
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(WORK_DIR, exist_ok=True)

# Reference numbers from the V5 champion (best epoch picked on the same val set)
CHAMPION = {"top1": 0.5338, "macro_recall": 0.5315, "n": 444}
print("✅ Settings loaded for experiment:", EXPERIMENT_NAME)

✅ Settings loaded for experiment: v7_dedup


## Cell 4: Data audit & cleaning (dedupe, leakage check, quality filter)

In [4]:
# ======================================================================
# CELL 4: DATA AUDIT & CLEANING  (dedupe -> leakage check -> quality filter)
# ======================================================================
import os, glob, json, pickle, hashlib, math
import numpy as np
from collections import Counter, defaultdict

EXPECTED_LABELS = {'bicep_curl': 0, 'high_knees': 1, 'jumping_jack': 2, 'lunge': 3,
                   'plank': 4, 'pushup': 5, 'squat': 6}
INV_LABELS = {v: k for k, v in EXPECTED_LABELS.items()}
NUM_CLASSES = 7
REQUIRED_KEYS = ['keypoint', 'keypoint_score', 'label', 'video_id', 'frame_dir',
                 'exercise', 'total_frames', 'img_shape']

# ---------------------------------------------------------------- helpers
def md5_file(path):
    h = hashlib.md5()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

def find_file(name):
    hits = []
    for root in SEARCH_ROOTS:
        hits += glob.glob(os.path.join(root, '**', name), recursive=True)
    if not hits and name.endswith(".pkl"):
        alt = name.replace("_1.pkl", ".pkl") if "_1.pkl" in name else name.replace(".pkl", "_1.pkl")
        for root in SEARCH_ROOTS:
            hits += glob.glob(os.path.join(root, '**', alt), recursive=True)
    hits = sorted(set(hits))
    if not hits:
        raise FileNotFoundError(f"{name} not found under {SEARCH_ROOTS}. Attach your dataset to the notebook.")
    if len(hits) > 1 and len({md5_file(h) for h in hits}) > 1:
        raise RuntimeError(f"Several DIFFERENT files named {name}: {hits}. Remove the extra one or rename.")
    return hits[0]

def content_hash(c):
    return hashlib.md5(np.ascontiguousarray(c['keypoint']).tobytes()).hexdigest()

def validate(clips, name):
    for c in clips:
        missing = [k for k in REQUIRED_KEYS if k not in c]
        if missing:
            raise ValueError(f"{name}: clip {c.get('frame_dir')} missing keys {missing}")
        kp = c['keypoint']
        if kp.ndim != 4 or kp.shape[0] != 1 or kp.shape[2:] != (17, 2):
            raise ValueError(f"{name}: unexpected keypoint shape {kp.shape} in {c['frame_dir']}")
        if EXPECTED_LABELS.get(c['exercise']) != c['label']:
            raise ValueError(f"{name}: label/exercise mismatch in {c['frame_dir']} "
                             f"({c['exercise']} vs {c['label']}). Class order must stay fixed.")

def clip_metrics(c):
    kp = c['keypoint'][0].astype(np.float64)
    sc = c['keypoint_score'][0]
    if not (np.isfinite(kp).all() and np.isfinite(sc).all()):
        return dict(finite=False, motion=0.0, fake=kp.shape[0], oob=1.0)
    d = np.abs(np.diff(kp, axis=0)).sum(axis=(1, 2))
    repeated = int((d == 0).sum())                                        # frame copied from previous
    placeholder = int((np.abs(kp - 500.0).max(axis=(1, 2)) < 1e-6).sum()) # all-500 filler frame
    lowvis = int((sc.mean(axis=1) < 0.3).sum())                           # almost no visible joints
    return dict(finite=True,
                motion=float(np.std(kp, axis=0).mean()),                  # TEMPORAL std per joint
                fake=max(repeated, placeholder, lowvis),
                oob=float(((kp < -50) | (kp > 1050)).mean()))

def frame_hashes(c):
    kp = np.round(c['keypoint'][0].astype(np.float32), 1)
    out = set()
    for i in range(kp.shape[0]):
        if np.abs(kp[i] - 500.0).max() < 1e-6:      # ignore placeholder frames
            continue
        out.add(hash(kp[i].tobytes()))
    return out

def per_class(clips):
    cnt = Counter(c['label'] for c in clips)
    return [cnt.get(i, 0) for i in range(NUM_CLASSES)]

def print_table(title, stages):
    print(f"\n{title}")
    print(f"{'class':<14}" + "".join(f"{name:>12}" for name, _ in stages))
    for i in range(NUM_CLASSES):
        print(f"{INV_LABELS[i]:<14}" + "".join(f"{v[i]:>12}" for _, v in stages))
    print(f"{'TOTAL':<14}" + "".join(f"{sum(v):>12}" for _, v in stages))

# ---------------------------------------------------------------- load
train_path = find_file(TRAIN_FILE_NAME)
val_path = find_file(VAL_FILE_NAME)
print("Train file:", train_path)
print("Val file  :", val_path)

with open(train_path, 'rb') as f:
    train_raw = pickle.load(f)
with open(val_path, 'rb') as f:
    val_raw = pickle.load(f)
validate(train_raw, "train")
validate(val_raw, "val")

report = {"train_file": train_path, "val_file": val_path, "removed": {}}
stages = [("raw", per_class(train_raw))]
print(f"\nRaw train clips: {len(train_raw)} | raw val clips: {len(val_raw)}")

# ---------------------------------------------------------------- 1) exact-duplicate removal (train)
train_hash = [content_hash(c) for c in train_raw]
lab_by_hash = defaultdict(set)
for h, c in zip(train_hash, train_raw):
    lab_by_hash[h].add(c['label'])
conflict = {h for h, s in lab_by_hash.items() if len(s) > 1}
seen, train = set(), []
for h, c in zip(train_hash, train_raw):
    if h in conflict or h in seen:
        continue
    seen.add(h)
    train.append(c)
report["removed"]["train_exact_duplicates"] = len(train_raw) - len(train) - sum(1 for h in train_hash if h in conflict)
report["removed"]["train_conflicting_label_copies"] = sum(1 for h in train_hash if h in conflict)
stages.append(("dedup", per_class(train)))

# ---------------------------------------------------------------- 2) val: keep full benchmark + unique-only copy
val_hash = [content_hash(c) for c in val_raw]
seen_v, val_unique = set(), []
for h, c in zip(val_hash, val_raw):
    if h not in seen_v:
        seen_v.add(h)
        val_unique.append(c)
report["val_full_clips"] = len(val_raw)
report["val_unique_clips"] = len(val_unique)
report["val_unique_per_class"] = per_class(val_unique)
val_hashes = set(val_hash)

# ---------------------------------------------------------------- 3) leakage removal (from TRAIN only)
n0 = len(train)
train = [c for c in train if content_hash(c) not in val_hashes]
report["removed"]["train_exact_match_with_val"] = n0 - len(train)

val_videos = {(c['label'], c['video_id']) for c in val_raw}
n0 = len(train)
train = [c for c in train if (c['label'], c['video_id']) not in val_videos]
report["removed"]["train_same_video_id_as_val"] = n0 - len(train)

idx = defaultdict(set)
for j, c in enumerate(val_unique):
    for fh in frame_hashes(c):
        idx[fh].add(j)
keep, near = [], 0
for c in train:
    cnt = Counter()
    for fh in frame_hashes(c):
        for j in idx.get(fh, ()):
            cnt[j] += 1
    if cnt and max(cnt.values()) >= NEAR_DUP_MIN_SHARED_FRAMES:
        near += 1
    else:
        keep.append(c)
train = keep
report["removed"]["train_near_duplicate_of_val"] = near
stages.append(("no leak", per_class(train)))

# ---------------------------------------------------------------- 4) quality filter (train only)
if APPLY_QUALITY_FILTER:
    reasons = Counter()
    keep = []
    for c in train:
        m = clip_metrics(c)
        if not m['finite']:
            reasons['nan_or_inf'] += 1
        elif m['motion'] < MIN_MOTION_OVERRIDES.get(INV_LABELS[c['label']], MIN_MOTION):
            reasons['low_motion'] += 1
        elif m['fake'] > MAX_FAKE_FRAMES:
            reasons['too_many_fake_frames'] += 1
        elif m['oob'] > MAX_OOB_FRACTION:
            reasons['out_of_frame'] += 1
        else:
            keep.append(c)
    train = keep
    report["removed"]["quality_filter"] = dict(reasons)
stages.append(("quality", per_class(train)))

# ---------------------------------------------------------------- 5) optional caps
if MAX_CLIPS_PER_VIDEO:
    by_vid = defaultdict(list)
    for c in train:
        by_vid[(c['label'], c['video_id'])].append(c)
    capped = []
    for v, lst in by_vid.items():
        lst = sorted(lst, key=lambda x: x['frame_dir'])
        if len(lst) > MAX_CLIPS_PER_VIDEO:
            pick = np.linspace(0, len(lst) - 1, MAX_CLIPS_PER_VIDEO).round().astype(int)
            lst = [lst[i] for i in sorted(set(pick.tolist()))]
        capped.extend(lst)
    report["removed"]["video_cap"] = len(train) - len(capped)
    train = capped
    stages.append(("capped", per_class(train)))

if TRAIN_FRACTION < 1.0:
    rng = np.random.RandomState(SEED)
    sub = []
    for lab in range(NUM_CLASSES):
        vids = sorted({c['video_id'] for c in train if c['label'] == lab})
        rng.shuffle(vids)
        chosen = set(vids[:max(1, math.ceil(TRAIN_FRACTION * len(vids)))])
        sub.extend(c for c in train if c['label'] == lab and c['video_id'] in chosen)
    report["removed"]["train_fraction"] = len(train) - len(sub)
    train = sub
    stages.append((f"frac{TRAIN_FRACTION}", per_class(train)))

# ---------------------------------------------------------------- final safety checks
train_counts = per_class(train)
assert min(train_counts) >= MIN_CLIPS_PER_CLASS, f"A class is too small after cleaning: {train_counts}"
assert not ({content_hash(c) for c in train} & val_hashes), "exact train/val overlap remains!"
assert not ({(c['label'], c['video_id']) for c in train} & val_videos), "video overlap remains!"
assert len({content_hash(c) for c in train}) == len(train), "duplicates remain in train!"

print_table("Training clips per class through each cleaning stage:", stages)
print("\nRemoved:", json.dumps(report["removed"], indent=2))
print(f"\nVal: {len(val_raw)} clips in the frozen file, only {len(val_unique)} unique "
      f"-> per class {report['val_unique_per_class']}")
imb = max(train_counts) / max(1, min(train_counts))
print(f"Class imbalance (max/min): {imb:.2f}")
print("✅ Integrity checks passed: no duplicates, no train/val overlap (content, video id, shared frames).")

# ---------------------------------------------------------------- class weights + save
if USE_CLASS_WEIGHTS:
    total = sum(train_counts)
    CLASS_WEIGHTS = [round(total / (NUM_CLASSES * n), 4) for n in train_counts]
else:
    CLASS_WEIGHTS = None
report["train_clips_final"] = len(train)
report["train_per_class_final"] = train_counts
report["class_weights"] = CLASS_WEIGHTS

TRAIN_PKL      = os.path.join(DATA_DIR, "train_clean.pkl")
VAL_UNIQUE_PKL = os.path.join(DATA_DIR, "val_unique.pkl")
VAL_FULL_PKL   = os.path.join(DATA_DIR, "val_full.pkl")
for path, obj in [(TRAIN_PKL, train), (VAL_UNIQUE_PKL, val_unique), (VAL_FULL_PKL, val_raw)]:
    with open(path, 'wb') as f:
        pickle.dump(obj, f)
with open(os.path.join(DATA_DIR, "data_report.json"), "w") as f:
    json.dump(report, f, indent=2)
with open(os.path.join(DATA_DIR, "class_order.json"), "w") as f:
    json.dump(EXPECTED_LABELS, f, indent=2)
print("Saved cleaned data to", DATA_DIR)

Train file: /kaggle/input/datasets/abdullahej/biomechai-v7-data/custom_dataset_train_v7.pkl
Val file  : /kaggle/input/datasets/abdullahej/biomechai-v7-data/custom_dataset_val_frozen.pkl

Raw train clips: 4434 | raw val clips: 444

Training clips per class through each cleaning stage:
class                  raw       dedup     no leak     quality
bicep_curl             582         450         449         408
high_knees             539         453         453         425
jumping_jack           585         467         467         423
lunge                  763         615         615         573
plank                  736         532         531         491
pushup                 582         422         420         381
squat                  647         508         508         458
TOTAL                 4434        3447        3443        3159

Removed: {
  "train_exact_duplicates": 987,
  "train_conflicting_label_copies": 0,
  "train_exact_match_with_val": 4,
  "train_same_video_id_as_val

## Cell 5: Train

In [5]:
# ======================================================================
# CELL 5: TRAIN POSEC3D v7  (V5 recipe, cleaned data, fixed seed)
# ======================================================================
import os, sys, subprocess

# --- custom augmentation (same as V5) ---
transform_code = '''import numpy as np
from mmcv.transforms import BaseTransform
from mmaction.registry import TRANSFORMS

@TRANSFORMS.register_module()
class RandomRotateKeypoints(BaseTransform):
    def __init__(self, max_angle=12.0, prob=0.5):
        self.max_angle = max_angle
        self.prob = prob

    def transform(self, results):
        if np.random.rand() > self.prob:
            return results

        angle = np.random.uniform(-self.max_angle, self.max_angle)
        rad = np.deg2rad(angle)
        cos_a, sin_a = np.cos(rad), np.sin(rad)

        h, w = results['img_shape'][:2]
        cx, cy = w / 2.0, h / 2.0

        kps = results['keypoint']
        x = kps[..., 0] - cx
        y = kps[..., 1] - cy

        new_x = cos_a * x - sin_a * y + cx
        new_y = sin_a * x + cos_a * y + cy

        new_x = np.clip(new_x, 0, w - 1)
        new_y = np.clip(new_y, 0, h - 1)

        results['keypoint'] = np.stack([new_x, new_y], axis=-1).astype(kps.dtype)
        return results
'''
with open(os.path.join(WORK_ROOT, "pose_transforms_extra.py"), "w") as f:
    f.write(transform_code)

# --- config (identical model / pipeline to V5; only data, seed, class weights differ) ---
if CLASS_WEIGHTS is not None:
    loss_cls_str = f"dict(type='CrossEntropyLoss', class_weight={CLASS_WEIGHTS})"
else:
    loss_cls_str = "dict(type='CrossEntropyLoss')"

config_template = """default_scope = 'mmaction'

load_from = 'https://download.openmmlab.com/mmaction/v1.0/skeleton/posec3d/slowonly_r50_8xb16-u48-240e_gym-limb/slowonly_r50_8xb16-u48-240e_gym-limb_20220815-2e6e3c5c.pth'

custom_imports = dict(imports=['pose_transforms_extra'], allow_failed_imports=True)

randomness = dict(seed=__SEED__, deterministic=False)

model = dict(
    type='Recognizer3D',
    backbone=dict(
        type='ResNet3dSlowOnly',
        depth=50,
        pretrained=None,
        in_channels=17,
        base_channels=32,
        num_stages=3,
        out_indices=(2,),
        stage_blocks=(4, 6, 3),
        conv1_stride_s=1,
        pool1_stride_s=1,
        inflate=(0, 1, 1),
        spatial_strides=(2, 2, 2),
        temporal_strides=(1, 1, 2),
        dilations=(1, 1, 1)),
    cls_head=dict(
        type='I3DHead',
        in_channels=512,
        num_classes=7,
        dropout_ratio=0.6,
        average_clips='prob',
        loss_cls=__LOSS_CLS__))

dataset_type = 'PoseDataset'
data_root = './'
ann_file_train = '__TRAIN_PKL__'
ann_file_val = '__VAL_PKL__'

left_kp = [1, 3, 5, 7, 9, 11, 13, 15]
right_kp = [2, 4, 6, 8, 10, 12, 14, 16]
skeletons = [
    [0, 5], [0, 6], [5, 7], [7, 9], [6, 8], [8, 10], [5, 11],
    [11, 13], [13, 15], [6, 12], [12, 14], [14, 16], [0, 1], [0, 2],
    [1, 3], [2, 4], [11, 12]
]

train_pipeline = [
    dict(type='UniformSampleFrames', clip_len=48),
    dict(type='PoseDecode'),
    dict(type='PoseCompact', hw_ratio=1., allow_imgpad=True),
    dict(type='Resize', scale=(-1, 64)),
    dict(type='RandomResizedCrop', area_range=(0.56, 1.0)),
    dict(type='Resize', scale=(56, 56), keep_ratio=False),
    dict(type='Flip', flip_ratio=0.5, left_kp=left_kp, right_kp=right_kp),
    dict(type='RandomRotateKeypoints', max_angle=12.0, prob=0.5),
    dict(type='GeneratePoseTarget', sigma=0.6, use_score=True, with_kp=False, with_limb=True, skeletons=skeletons),
    dict(type='FormatShape', input_format='NCTHW_Heatmap'),
    dict(type='PackActionInputs')
]

val_pipeline = [
    dict(type='UniformSampleFrames', clip_len=48, num_clips=1, test_mode=True),
    dict(type='PoseDecode'),
    dict(type='PoseCompact', hw_ratio=1., allow_imgpad=True),
    dict(type='Resize', scale=(-1, 56)),
    dict(type='CenterCrop', crop_size=56),
    dict(type='GeneratePoseTarget', sigma=0.6, use_score=True, with_kp=False, with_limb=True, skeletons=skeletons),
    dict(type='FormatShape', input_format='NCTHW_Heatmap'),
    dict(type='PackActionInputs')
]

train_dataloader = dict(
    batch_size=__BATCH__,
    num_workers=2,
    persistent_workers=True,
    sampler=dict(type='DefaultSampler', shuffle=True),
    dataset=dict(
        type=dataset_type,
        ann_file=ann_file_train,
        pipeline=train_pipeline))

val_dataloader = dict(
    batch_size=__BATCH__,
    num_workers=2,
    persistent_workers=True,
    sampler=dict(type='DefaultSampler', shuffle=False),
    dataset=dict(
        type=dataset_type,
        ann_file=ann_file_val,
        pipeline=val_pipeline,
        test_mode=True))

test_dataloader = dict(
    batch_size=__BATCH__,
    num_workers=2,
    persistent_workers=True,
    sampler=dict(type='DefaultSampler', shuffle=False),
    dataset=dict(
        type=dataset_type,
        ann_file=ann_file_val,
        pipeline=val_pipeline,
        test_mode=True))

optim_wrapper = dict(
    optimizer=dict(type='SGD', lr=__LR__, momentum=0.9, weight_decay=0.0005),
    clip_grad=dict(max_norm=40, norm_type=2))

train_cfg = dict(type='EpochBasedTrainLoop', max_epochs=__EPOCHS__, val_interval=__VALINT__)
val_cfg = dict(type='ValLoop')
val_evaluator = [dict(type='AccMetric')]
test_cfg = dict(type='TestLoop')
test_evaluator = [dict(type='AccMetric')]

default_hooks = dict(
    checkpoint=dict(type='CheckpointHook', interval=2, max_keep_ckpts=3, save_best='acc/top1'),
    logger=dict(type='LoggerHook', interval=20))
"""
config_code = (config_template
    .replace("__SEED__", str(SEED))
    .replace("__LOSS_CLS__", loss_cls_str)
    .replace("__TRAIN_PKL__", TRAIN_PKL)
    .replace("__VAL_PKL__", VAL_UNIQUE_PKL)
    .replace("__BATCH__", str(BATCH_SIZE))
    .replace("__LR__", str(LR))
    .replace("__EPOCHS__", str(MAX_EPOCHS))
    .replace("__VALINT__", str(VAL_INTERVAL)))
with open(CONFIG_PY, "w") as f:
    f.write(config_code)
print("✅ Config written:", CONFIG_PY)
print("   train clips:", report["train_clips_final"], "| val (model selection): unique val clips only")

# --- launch training ---
env = os.environ.copy()
env["PYTHONPATH"] = f"{WORK_ROOT}:/kaggle/working/mmaction2:{env.get('PYTHONPATH', '')}"
train_cmd = [sys.executable, "/kaggle/working/mmaction2/tools/train.py", CONFIG_PY, f"--work-dir={WORK_DIR}"]

print("=" * 70)
print(f"🚀 TRAINING POSEC3D {EXPERIMENT_NAME} ({MAX_EPOCHS} epochs)")
print("=" * 70)
proc = subprocess.Popen(train_cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
all_output = []
for line in iter(proc.stdout.readline, ''):
    sys.stdout.write(line)
    sys.stdout.flush()
    all_output.append(line)
proc.wait()
if proc.returncode != 0:
    print("".join(all_output[-30:]))
    raise RuntimeError(f"Training failed with return code {proc.returncode}")
print("\n✅ Training completed successfully!")

✅ Config written: /kaggle/working/posec3d_biomechai_v7_dedup.py
   train clips: 3159 | val (model selection): unique val clips only
🚀 TRAINING POSEC3D v7_dedup (18 epochs)
10/07 06:52:35 - mmengine - INFO - 
------------------------------------------------------------
System environment:
    sys.platform: linux
    Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
    CUDA available: True
    MUSA available: False
    numpy_random_seed: 42
    GPU 0,1: Tesla T4
    CUDA_HOME: /usr/local/cuda
    NVCC: Cuda compilation tools, release 12.8, V12.8.93
    GCC: x86_64-linux-gnu-gcc (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0
    PyTorch: 2.11.0+cu128
    PyTorch compiling details: PyTorch built with:
  - GCC 13.3
  - C++ Version: 201703
  - Intel(R) oneAPI Math Kernel Library Version 2024.2-Product Build 20240605 for Intel(R) 64 architecture applications
  - Intel(R) MKL-DNN v3.10.2 (Git Hash f1d471933dc852f956fd05389f9313c7148783d5)
  - OpenMP 201511 (a.k.a. OpenMP 4.5)
  - LAPACK is 

## Cell 6: Evaluate best and last checkpoint

In [6]:
# ======================================================================
# CELL 6: EVALUATE best AND last checkpoint on the unique-clip val set
#         and on the full 444-clip frozen val file
# ======================================================================
import os, re, sys, glob, json, pickle, math, subprocess
import numpy as np
import torch

env = os.environ.copy()
env["PYTHONPATH"] = f"{WORK_ROOT}:/kaggle/working/mmaction2:{env.get('PYTHONPATH', '')}"

best_list = sorted(glob.glob(os.path.join(WORK_DIR, "best_acc_top1_epoch_*.pth")))
epoch_list = sorted(glob.glob(os.path.join(WORK_DIR, "epoch_*.pth")),
                    key=lambda p: int(re.search(r"epoch_(\d+)", p).group(1)))
best_ckpt = best_list[-1] if best_list else epoch_list[-1]
last_ckpt = epoch_list[-1]
print("Best checkpoint:", os.path.basename(best_ckpt))
print("Last checkpoint:", os.path.basename(last_ckpt))

def run_test(ckpt, ann_file, tag):
    dump = os.path.join(WORK_ROOT, f"result_{EXPERIMENT_NAME}_{tag}.pkl")
    cmd = [sys.executable, "/kaggle/working/mmaction2/tools/test.py", CONFIG_PY, ckpt,
           f"--dump={dump}", "--cfg-options", f"test_dataloader.dataset.ann_file={ann_file}"]
    r = subprocess.run(cmd, env=env, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr[-3000:])
        raise RuntimeError(f"test.py failed for {tag}")
    return dump

def to_np(x):
    return x.detach().cpu().numpy() if hasattr(x, "detach") else np.asarray(x)

def load_preds(dump):
    with open(dump, "rb") as f:
        res = pickle.load(f)
    y_true, y_pred = [], []
    for r in res:
        score = to_np(r["pred_score"]).reshape(-1)
        gt = int(to_np(r["gt_label"]).reshape(-1)[0])
        y_pred.append(int(np.argmax(score)))
        y_true.append(gt)
    return np.array(y_true), np.array(y_pred)

def wilson(k, n, z=1.96):
    if n == 0:
        return (0.0, 0.0)
    p = k / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return (c - h, c + h)

def summarize(y_true, y_pred):
    n = len(y_true)
    k = int((y_true == y_pred).sum())
    recalls = []
    for i in range(NUM_CLASSES):
        m = y_true == i
        recalls.append(float((y_pred[m] == i).mean()) if m.any() else float("nan"))
    cm = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=int)
    for t, p in zip(y_true, y_pred):
        cm[t, p] += 1
    lo, hi = wilson(k, n)
    return dict(n=n, correct=k, top1=k / n, ci95=[lo, hi],
                macro_recall=float(np.nanmean(recalls)), per_class_recall=recalls,
                support=[int((y_true == i).sum()) for i in range(NUM_CLASSES)],
                confusion=cm.tolist())

results = {}
for ck_name, ck in [("best", best_ckpt), ("last", last_ckpt)]:
    for set_name, ann in [("val_unique", VAL_UNIQUE_PKL), ("val_full", VAL_FULL_PKL)]:
        tag = f"{ck_name}_{set_name}"
        dump = run_test(ck, ann, tag)
        yt, yp = load_preds(dump)
        results[tag] = summarize(yt, yp)

print("\n" + "=" * 78)
print(f"{'checkpoint / val set':<26}{'n':>6}{'top-1':>10}{'95% CI':>20}{'macro recall':>16}")
print("-" * 78)
for tag, s in results.items():
    print(f"{tag:<26}{s['n']:>6}{s['top1']*100:>9.2f}%"
          f"   [{s['ci95'][0]*100:5.1f}, {s['ci95'][1]*100:5.1f}]{s['macro_recall']*100:>14.2f}%")
print("-" * 78)
print(f"Champion V5 (reference, 444-row val, best epoch picked on val): "
      f"{CHAMPION['top1']*100:.2f}% top-1, {CHAMPION['macro_recall']*100:.2f}% macro recall")
print("=" * 78)

print("\nPer-class recall (last checkpoint, unique val):")
s = results["last_val_unique"]
for i in range(NUM_CLASSES):
    print(f"  {INV_LABELS[i]:<14} recall {s['per_class_recall'][i]*100:5.1f}%   (n={s['support'][i]})")
print("\nConfusion matrix (rows = true, cols = predicted), last checkpoint, unique val:")
print("  order:", [INV_LABELS[i] for i in range(NUM_CLASSES)])
for i, row in enumerate(s["confusion"]):
    print(f"  {INV_LABELS[i]:<14}", row)

print("\nHow to read this:")
print(" - 'best' was chosen using val_unique, so it is optimistic on that set.")
print(" - 'last' was not selected on val, so it is the fairer number.")
print(" - Differences smaller than the CI width are not meaningful.")

with open(os.path.join(WORK_ROOT, f"results_{EXPERIMENT_NAME}.json"), "w") as f:
    json.dump(dict(best_ckpt=os.path.basename(best_ckpt), last_ckpt=os.path.basename(last_ckpt),
                   results=results, champion_reference=CHAMPION), f, indent=2)

Best checkpoint: best_acc_top1_epoch_8.pth
Last checkpoint: epoch_18.pth

checkpoint / val set           n     top-1              95% CI    macro recall
------------------------------------------------------------------------------
best_val_unique              192    66.15%   [ 59.2,  72.5]         63.28%
best_val_full                444    58.78%   [ 54.1,  63.3]         56.58%
last_val_unique              192    60.42%   [ 53.4,  67.1]         63.01%
last_val_full                444    51.80%   [ 47.2,  56.4]         53.12%
------------------------------------------------------------------------------
Champion V5 (reference, 444-row val, best epoch picked on val): 53.38% top-1, 53.15% macro recall

Per-class recall (last checkpoint, unique val):
  bicep_curl     recall  35.9%   (n=39)
  high_knees     recall  73.7%   (n=19)
  jumping_jack   recall  51.5%   (n=33)
  lunge          recall  84.0%   (n=25)
  plank          recall  67.9%   (n=28)
  pushup         recall  62.5%   (n=16)
  

## Cell 7: Package artifacts

In [7]:
# ======================================================================
# CELL 7: PACKAGE ARTIFACTS (download from the Kaggle Output tab / link below)
# ======================================================================
import os, glob, zipfile
from IPython.display import FileLink, display

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(CONFIG_PY, arcname=os.path.basename(CONFIG_PY))
    z.write(os.path.join(WORK_ROOT, f"results_{EXPERIMENT_NAME}.json"), arcname="results.json")
    z.write(os.path.join(DATA_DIR, "data_report.json"), arcname="data_report.json")
    z.write(os.path.join(DATA_DIR, "class_order.json"), arcname="class_order.json")
    z.write(best_ckpt, arcname="checkpoints/" + os.path.basename(best_ckpt))
    if last_ckpt != best_ckpt:
        z.write(last_ckpt, arcname="checkpoints/" + os.path.basename(last_ckpt))
    for pat in ("**/*.log", "**/scalars.json"):
        for p in glob.glob(os.path.join(WORK_DIR, pat), recursive=True):
            z.write(p, arcname="logs/" + os.path.relpath(p, WORK_DIR).replace(os.sep, "_"))

print(f"📦 Created {ZIP_PATH} ({os.path.getsize(ZIP_PATH) / 1024**2:.2f} MB)")
display(FileLink(os.path.basename(ZIP_PATH)))

📦 Created /kaggle/working/BioMechAI_PoseC3D_v7_dedup_artifacts.zip (21.60 MB)


/kaggle/working/BioMechAI_PoseC3D_v7_dedup_artifacts.zip

## How to interpret the results
- Compare the **last** checkpoint (not selected on val) and the **best** checkpoint against the champion's 53.38%. The champion number was also a best-epoch pick on the same val data, so the fairest comparison is best-vs-best, plus last-vs-champion as a sanity check.
- The val set has only **192 unique clips**, so the 95% interval is roughly ±7 points. A change smaller than that is not evidence of improvement.
- Check the per-class recall and confusion matrix: if the errors are concentrated in pairs like squat/lunge or high_knees/jumping_jack, more data alone will help less than targeted data or features.

## Using the new weights in the app
The model definition is identical to V5 (17 keypoints, 48 frames, 56×56, 7 classes), so the checkpoint is a drop-in replacement. Keep the class order in `class_order.json` and re-check any confidence thresholds, since probabilities from a different model are distributed differently.

## Useful experiments (change Cell 3, rename `EXPERIMENT_NAME`)
- `TRAIN_FRACTION = 0.5` and `0.25`: a learning curve to see whether more data is still helping.
- `USE_CLASS_WEIGHTS = False`: check whether class weighting matters.
- `MAX_CLIPS_PER_VIDEO = 6`: stop a few videos from dominating a class.
